# H1: coarse-to-fine prefixes (sentence level)

This notebook tests H1 from [H1_plan.md](H1_plan.md), step by step.

**Definition used: label hierarchy.** "Coarse" and "fine" are two levels of the same label hierarchy on the same sentences. Each fine label belongs to exactly one coarse label.

**Steps in this version**
1. Load the two datasets and their label levels (TREC, GoEmotions).
2. Load the models (base TinyBERT and the 12 trained checkpoints).
3. Extract the sentence vectors once and save them.
4. Build the coordinate orders (prefix, reverse, 10 random, PCA).
5. Fit the probes and report test macro-F1.
6. Calculate the metrics of plan §7 (R, A, ΔCFI, paired bootstrap) and apply the decision rules of plan §8.
7. Analysis.

## 0. Setup

- `REPO_ROOT` is the repository folder (two levels above this `experiments/coarse_to_fine/` folder). We add it to `sys.path` so we can import the repository's own code, such as `h11_probe.load_encoder`.
- `CHECKPOINT_ROOT` is where the trained weights are.
- `HF_HUB_DISABLE_SYMLINKS_WARNING` only silences a harmless Windows warning from the Hugging Face cache.

In [44]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

REPO_ROOT = Path.cwd().parents[1]        # this notebook lives in <repo>/experiments/coarse_to_fine/
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CHECKPOINT_ROOT = REPO_ROOT / "MRL_LEXIS_H11_checkpoints" / "h11_probe" / "runs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Repository :", REPO_ROOT)
print("Checkpoints:", CHECKPOINT_ROOT, "| exists:", CHECKPOINT_ROOT.exists())
print("Device     :", DEVICE)

Repository : c:\Users\Eshwar\Desktop\Course files\NLP\MIPIC\MIPIC_A3\assignment-3-mipic
Checkpoints: c:\Users\Eshwar\Desktop\Course files\NLP\MIPIC\MIPIC_A3\assignment-3-mipic\MRL_LEXIS_H11_checkpoints\h11_probe\runs | exists: True
Device     : cpu


## 1. Datasets

### 1a. TREC (question classification)

- About 6,000 short questions, labelled by the kind of answer they ask for.
- Two label levels: **6 coarse** classes (e.g. `LOC`) and **50 fine** classes (e.g. `LOC:city`).
- Splits: 5,452 train, 500 test. There is no official validation split (we will make one later from train).

In [45]:
from datasets import load_dataset

trec_raw = load_dataset("CogComp/trec", revision="refs/convert/parquet")
TREC_COARSE_NAMES = trec_raw["train"].features["coarse_label"].names
TREC_FINE_NAMES = trec_raw["train"].features["fine_label"].names


def trec_to_frame(split):
    # One row per question: the text, the integer labels and their names.
    frame = trec_raw[split].to_pandas()
    frame = frame.rename(columns={"coarse_label": "coarse", "fine_label": "fine"})
    frame["coarse_name"] = frame["coarse"].map(lambda i: TREC_COARSE_NAMES[i])
    frame["fine_name"] = frame["fine"].map(lambda i: TREC_FINE_NAMES[i])
    return frame


trec = {split: trec_to_frame(split) for split in ("train", "test")}

# Check the hierarchy: the fine name must start with "<coarse name>:".
for split, frame in trec.items():
    nested = (frame["fine_name"].str.split(":").str[0] == frame["coarse_name"]).all()
    assert nested, f"TREC {split}: a fine label is not inside its coarse label"

print({split: len(frame) for split, frame in trec.items()})
print("coarse classes:", len(TREC_COARSE_NAMES), "| fine classes:", len(TREC_FINE_NAMES))
trec["train"].head()

{'train': 5452, 'test': 500}
coarse classes: 6 | fine classes: 50


,text,coarse,fine,coarse_name,fine_name
0,How did serfdom develop in and then leave Russ...,2,26,DESC,DESC:manner
1,What films featured the character Popeye Doyle ?,1,5,ENTY,ENTY:cremat
2,How can I find a list of celebrities ' real na...,2,26,DESC,DESC:manner
3,What fowl grabs the spotlight after the Chines...,1,2,ENTY,ENTY:animal
4,What is the full form of .com ?,0,1,ABBR,ABBR:exp


How the classes are spread. Note how few test questions some fine classes have; fine-level scores on TREC will be noisy.

In [46]:
print("Coarse classes (train / test):")
print(pd.concat({s: trec[s]["coarse_name"].value_counts() for s in trec}, axis=1).fillna(0).astype(int))

fine_test_counts = trec["test"]["fine_name"].value_counts()
print("\nFine classes present in test:", fine_test_counts.size, "of", len(TREC_FINE_NAMES))
print("Fine classes with fewer than 5 test questions:", int((fine_test_counts < 5).sum()))

Coarse classes (train / test):
             train  test
coarse_name             
ENTY          1250    94
HUM           1223    65
DESC          1162   138
NUM            896   113
LOC            835    81
ABBR            86     9

Fine classes present in test: 42 of 50
Fine classes with fewer than 5 test questions: 19


### 1b. GoEmotions (emotions in Reddit comments)

- About 58,000 Reddit comments with **28 labels** (27 emotions plus `neutral`).
- Splits ("simplified" version): 43,410 train, 5,426 validation, 5,427 test.
- A comment can have more than one label. **We keep only comments with exactly one label**, so that each comment has one fine, one Ekman and one sentiment label.

**The hierarchy** comes from the dataset's official mapping files (`ekman_mapping.json` and `sentiment_mapping.json` in the GoEmotions GitHub repository):

- fine (28) → **Ekman** (7): anger, disgust, fear, joy, sadness, surprise, neutral
- Ekman (7) → **sentiment** (4): positive, negative, ambiguous, neutral

We write the mapping as two small dictionaries. Building sentiment from Ekman (not separately from fine) makes the three levels nested by construction.

In [47]:
go_raw = load_dataset("google-research-datasets/go_emotions", "simplified")
GO_FINE_NAMES = go_raw["train"].features["labels"].feature.names   # 28 names; index = label id

# Official Ekman mapping (fine emotion -> Ekman group). "neutral" maps to itself.
EKMAN_GROUPS = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude", "love",
                 "optimism", "relief", "pride", "admiration", "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"],
}
# Official sentiment mapping, written at the Ekman level.
EKMAN_TO_SENTIMENT = {
    "joy": "positive",
    "anger": "negative", "disgust": "negative", "fear": "negative", "sadness": "negative",
    "surprise": "ambiguous",
    "neutral": "neutral",
}

FINE_TO_EKMAN = {fine: ekman for ekman, fines in EKMAN_GROUPS.items() for fine in fines}
assert sorted(FINE_TO_EKMAN) == sorted(GO_FINE_NAMES), "every fine emotion needs exactly one Ekman group"

GO_EKMAN_NAMES = list(EKMAN_GROUPS)                                   # 7
GO_SENTIMENT_NAMES = ["positive", "negative", "ambiguous", "neutral"]  # 4


def go_to_frame(split):
    frame = go_raw[split].to_pandas()
    frame = frame[frame["labels"].map(len) == 1].copy()     # keep single-label comments only
    frame["fine_name"] = frame["labels"].map(lambda labels: GO_FINE_NAMES[labels[0]])
    frame["ekman_name"] = frame["fine_name"].map(FINE_TO_EKMAN)
    frame["sentiment_name"] = frame["ekman_name"].map(EKMAN_TO_SENTIMENT)
    # Integer labels for the probes later.
    frame["fine"] = frame["fine_name"].map(GO_FINE_NAMES.index)
    frame["ekman"] = frame["ekman_name"].map(GO_EKMAN_NAMES.index)
    frame["sentiment"] = frame["sentiment_name"].map(GO_SENTIMENT_NAMES.index)
    return frame[["id", "text", "fine", "fine_name", "ekman", "ekman_name",
                  "sentiment", "sentiment_name"]].reset_index(drop=True)


goemo = {split: go_to_frame(split) for split in ("train", "validation", "test")}

for split in goemo:
    print(f"{split:10s} kept {len(goemo[split]):6d} of {len(go_raw[split]):6d} comments (single label)")
goemo["train"].head()

train      kept  36308 of  43410 comments (single label)
validation kept   4548 of   5426 comments (single label)
test       kept   4590 of   5427 comments (single label)


,id,text,fine,fine_name,ekman,ekman_name,sentiment,sentiment_name
0,eebbqej,My favourite food is anything I didn't have to...,27,neutral,6,neutral,3,neutral
1,ed00q6i,"Now if he does off himself, everyone will thin...",27,neutral,6,neutral,3,neutral
2,eezlygj,WHY THE FUCK IS BAYLESS ISOING,2,anger,0,anger,1,negative
3,ed7ypvh,To make her feel threatened,14,fear,2,fear,1,negative
4,ed0bdzj,Dirty Southern Wankers,3,annoyance,0,anger,1,negative


How the classes are spread at each level (train split). The classes are very unbalanced: `neutral` is the largest fine emotion, and several fine emotions (such as `grief`) are rare.

In [48]:
for level in ("sentiment_name", "ekman_name", "fine_name"):
    counts = goemo["train"][level].value_counts()
    print(f"{level} ({counts.size} classes): largest {counts.index[0]} = {counts.iloc[0]}, "
          f"smallest {counts.index[-1]} = {counts.iloc[-1]}")

sentiment_name (4 classes): largest positive = 12920, smallest ambiguous = 3553
ekman_name (7 classes): largest joy = 12920, smallest disgust = 498
fine_name (28 classes): largest neutral = 12823, smallest grief = 39


## 2. Models

### What we load
- **base**: pretrained `huawei-noah/TinyBERT_General_6L_768D`, no MIPIC training. Its pooler is random (Finding 9), so it is our "no ordering" null model.
- **4 trained arms × 3 seeds** from the friend's runs (attention KL **off**): `mrl`, `mrl_pic`, `mrl_sia`, `mipic`; seeds 42, 1, 2.

Each run folder has two files:
- `best.pt`: the epoch picked by the training script's held-out probe (epoch 1 to 3). **We use this one**; the earlier `h11_probe` results are based on it.
- `model.pt`: the model after the last epoch (5).

### How loading works
`h11_probe.load_encoder.load_encoder(path)` (already in the repository, with tests):
1. downloads the base TinyBERT and its tokenizer from Hugging Face (cached after the first time);
2. reads the checkpoint and keeps only the encoder weights; the training-only parts (SIA projections, PIC heads) are dropped, since we do not need them to make embeddings;
3. loads them with `strict=True`, so a missing or extra weight raises an error instead of failing silently;
4. returns `(encoder, tokenizer)` in eval mode.

`path=None` gives the base model.

In [49]:
from experiments.layerwise_probe.load_encoder import load_encoder

ARMS = ["mrl", "mrl_pic", "mrl_sia", "mipic"]
SEEDS = [42, 1, 2]

# Model name -> checkpoint path (None = base model).
CHECKPOINTS = {"base": None}
for seed in SEEDS:
    for arm in ARMS:
        CHECKPOINTS[f"{arm}_s{seed}"] = CHECKPOINT_ROOT / f"seed{seed}" / "checkpoints" / arm / "best.pt"

missing = [name for name, path in CHECKPOINTS.items() if path is not None and not path.exists()]
print(f"{len(CHECKPOINTS)} models listed; missing files: {missing or 'none'}")

13 models listed; missing files: none


A small helper loads one model by name. We load models **one at a time** when we need them, because each one takes about 270 MB of memory.

In [50]:
def load_model(name):
    # Return (encoder, tokenizer) for a name in CHECKPOINTS, e.g. "base" or "mipic_s42".
    return load_encoder(CHECKPOINTS[name], device=DEVICE)

### Check: load two models and embed two sentences

The sentence embedding z is `pooler_output`: **one 768-value vector per sentence** (shape `[batch, 768]`). Its first d values are the width-d prefix that H1 is about.

If loading works, the base and MIPIC vectors for the same sentence should differ (training changed the encoder).

In [51]:
sentences = ["What is the capital of Peru?", "Thank you so much, this made my day!"]


@torch.no_grad()
def embed(encoder, tokenizer, texts):
    batch = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    return encoder(**batch).pooler_output.float().cpu()      # shape [len(texts), 768]


z = {}
for name in ("base", "mipic_s42"):
    encoder, tokenizer = load_model(name)
    z[name] = embed(encoder, tokenizer, sentences)
    del encoder                                              # free memory before the next model

print("shape:", tuple(z["base"].shape))
print("first 5 values of z, sentence 1, base     :", z["base"][0, :5].numpy().round(4))
print("first 5 values of z, sentence 1, mipic_s42:", z["mipic_s42"][0, :5].numpy().round(4))
print("cosine(base, mipic_s42) per sentence      :",
      torch.nn.functional.cosine_similarity(z["base"], z["mipic_s42"]).numpy().round(3))

load_encoder: best.pt [wrapper ('core.encoder.' prefix)] loaded 103 keys, dropped 26 non-encoder keys
shape: (2, 768)
first 5 values of z, sentence 1, base     : [-0.0161  0.0004  0.0732  0.0298 -0.0638]
first 5 values of z, sentence 1, mipic_s42: [-0.0853 -0.0567  0.0605  0.0121 -0.128 ]
cosine(base, mipic_s42) per sentence      : [0.112 0.272]


## 3. Features

Each model reads each sentence once. We save two vectors for each sentence:

- `pooler`: `pooler_output`. This is z, the **main** read-out (plan §3).
- `cls6`: the raw layer-6 [CLS] vector. This is the **secondary** read-out, which SIA and PIC act on.

The files go to `experiments/coarse_to_fine/features/<model>__<dataset>__<split>.npz` (float32). A file that already exists is skipped, so you can stop the cell and start it again later.

**Speed.** The sentences are sorted by length, so each batch has little padding. Then the vectors are put back in the original order.

**Splits.** We extract only `train` and `test`. The probes choose C by 5-fold CV on train (plan §6), so they do not need a validation split.

In [52]:
import time

FEATURE_DIR = Path.cwd() / "features"
FEATURE_DIR.mkdir(exist_ok=True)

DATASETS = {"trec": trec, "goemo": goemo}
SPLITS = ["train", "test"]
BATCH_SIZE = 64


def feature_path(model, dataset, split):
    return FEATURE_DIR / f"{model}__{dataset}__{split}.npz"


@torch.no_grad()
def extract(encoder, tokenizer, texts):
    # Return (pooler, cls6): two float32 arrays [len(texts), 768], rows in the order of `texts`.
    pooler = np.empty((len(texts), 768), dtype=np.float32)
    cls6 = np.empty((len(texts), 768), dtype=np.float32)
    order = np.argsort([len(t) for t in texts])        # similar lengths together -> little padding
    for start in range(0, len(texts), BATCH_SIZE):
        idx = order[start:start + BATCH_SIZE]
        batch = tokenizer([texts[i] for i in idx], padding=True, truncation=True,
                          max_length=256, return_tensors="pt").to(DEVICE)
        out = encoder(**batch)
        pooler[idx] = out.pooler_output.float().cpu().numpy()
        cls6[idx] = out.last_hidden_state[:, 0].float().cpu().numpy()   # layer 6 = last layer
    return pooler, cls6


def extract_all(models):
    for name in models:
        todo = [(ds, s) for ds in DATASETS for s in SPLITS if not feature_path(name, ds, s).exists()]
        if not todo:
            print(f"{name:12s} features already saved")
            continue
        encoder, tokenizer = load_model(name)
        for ds, split in todo:
            start = time.time()
            texts = DATASETS[ds][split]["text"].tolist()
            pooler, cls6 = extract(encoder, tokenizer, texts)
            np.savez(feature_path(name, ds, split), pooler=pooler, cls6=cls6)
            print(f"{name:12s} {ds:5s} {split:5s} {len(texts):6d} sentences  {time.time() - start:6.1f} s")
        del encoder

`MODELS_TO_RUN` sets which models the next steps use. The default is all 13 (base + 4 arms × 3 seeds). For a quick first look, use a short list, for example `["base", "mrl_s42", "mipic_s42"]`.

In [53]:
MODELS_TO_RUN = list(CHECKPOINTS)
extract_all(MODELS_TO_RUN)

base         features already saved
mrl_s42      features already saved
mrl_pic_s42  features already saved
mrl_sia_s42  features already saved
mipic_s42    features already saved
mrl_s1       features already saved
mrl_pic_s1   features already saved
mrl_sia_s1   features already saved
mipic_s1     features already saved
mrl_s2       features already saved
mrl_pic_s2   features already saved
mrl_sia_s2   features already saved
mipic_s2     features already saved


## 4. Coordinate orders (plan §5)

An **order** is a list of the 768 coordinate numbers. At width d, the probe gets the first d coordinates of the order.

- `prefix`: 0, 1, 2, ... This is the MRL order.
- `reverse`: 767, 766, ... This gives the **last** d coordinates.
- `random_00` ... `random_09`: 10 random permutations with fixed seeds (1000 + k). Each one is nested in the same way as the prefix: its first 16 coordinates are inside its first 32.
- `pca`: the top d principal components of the model's own train vectors. This order is different for each model, so step 5 makes it.

**Width 768.** At d = 768, prefix, reverse and every random order hold the same 768 coordinates, so their probes are the same. We fit d = 768 once and call it `full`.

The random orders are saved to `features/random_orders.npy`, so later steps (the bootstrap) use the same sets.

In [54]:
D_FULL = 768
WIDTHS = [16, 32, 64, 128, 256]        # d < 768; d = 768 is fitted once as "full"
N_RANDOM = 10                          # random order k always uses seed 1000 + k

ORDERS = {"prefix": np.arange(D_FULL), "reverse": np.arange(D_FULL)[::-1].copy()}
for k in range(N_RANDOM):
    ORDERS[f"random_{k:02d}"] = np.random.default_rng(1000 + k).permutation(D_FULL)

np.save(FEATURE_DIR / "random_orders.npy", np.stack([ORDERS[f"random_{k:02d}"] for k in range(N_RANDOM)]))
print(f"{len(ORDERS)} fixed orders (+ pca for each model)")
print("random_00, first 8 coordinates:", ORDERS["random_00"][:8])

12 fixed orders (+ pca for each model)
random_00, first 8 coordinates: [684 625 446 491 179  42 276 286]


## 5. Probes (plan §6)

For each **model × dataset × label level × read-out**, `probe_one` does these steps:

1. **Choose C.** Fit `StandardScaler + LogisticRegression` on all 768 coordinates of train, with 5-fold stratified CV, for each C in `C_GRID`. Keep the C with the best mean CV macro-F1. The scaler is fitted again inside each fold, so no fold sees the statistics of its held-out part.
2. **Standardize** the train and test vectors with the train mean and SD. Each coordinate is scaled on its own, so we can standardize all 768 once and then take any subset.
3. **PCA order.** Fit PCA on the train vectors of this model, then standardize the component scores.
4. **Logistic-regression probes.** Fit one probe for each order × width with the C from step 1, plus `full` (d = 768), then predict the test split. The fits run in parallel on all CPU cores.
5. **kNN check.** Use k = 10 with cosine distance, for prefix and the random orders at d = 16 and 32. This step fits no classifier. It uses the raw (unstandardized) coordinates, because it asks whether same-label sentences are near each other in z. kNN runs on only 4 cores with small distance blocks, because on GoEmotions it needs much more memory than logistic regression.

**Two F1 columns.** The handling of the TREC fine level is not decided yet (only 42 of 50 classes are in test). So we save both values and decide later:

- `f1_macro`: the scikit-learn default. It averages over every class that is in the test labels **or** the predictions.
- `f1_macro_test_classes`: the average over only the classes that are in the test labels.

On GoEmotions and on TREC coarse, all classes are in test, so the two values are almost always equal.

**Saved outputs** (in `experiments/coarse_to_fine/results/`):

- `probe_f1.csv`: one row per probe.
- `predictions/*.npz`: the test predictions of every probe, for the bootstrap later (plan §7).

Finished combinations are skipped when the cell runs again.

`converged` is False if logistic regression hit `MAX_ITER` before it converged. Check this column before you use a score.

In [55]:
from joblib import Parallel, delayed
from sklearn import config_context
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

LEVELS = {"trec": ["coarse", "fine"], "goemo": ["sentiment", "ekman", "fine"]}
READOUTS = ["pooler"]                  # add "cls6" for the secondary read-out
C_GRID = [0.01, 0.1, 1.0, 10.0]
MAX_ITER = 2000
KNN_K = 10
KNN_WIDTHS = [16, 32]
N_JOBS = -1                            # all CPU cores (logistic regression)
KNN_N_JOBS = 4                         # kNN needs far more memory per job, so fewer at once
KNN_WORKING_MEMORY_MB = 256            # size of each kNN distance block (sklearn default: 1024)

RESULT_DIR = Path.cwd() / "results"
PRED_DIR = RESULT_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CSV = RESULT_DIR / "probe_f1.csv"


def make_probe(C):
    return LogisticRegression(C=C, max_iter=MAX_ITER)


def select_C(X_train, y_train):
    # Plan §6: 5-fold CV on train at full width; the scaler is refitted inside each fold.
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    cv_f1 = {}
    for C in C_GRID:
        pipe = make_pipeline(StandardScaler(), make_probe(C))
        cv_f1[C] = cross_val_score(pipe, X_train, y_train, cv=folds,
                                   scoring="f1_macro", n_jobs=N_JOBS).mean()
    return max(cv_f1, key=cv_f1.get), cv_f1


def fit_logreg(X_train, X_test, cols, y_train, C):
    probe = make_probe(C).fit(X_train[:, cols], y_train)
    return probe.predict(X_test[:, cols]), bool(probe.n_iter_.max() < MAX_ITER)


def fit_knn(X_train, X_test, cols, y_train):
    # Brute-force kNN compares blocks of test rows with all train rows (36k on GoEmotions).
    # With the default 1 GB blocks, 20 parallel jobs ran out of RAM; 256 MB x 4 jobs is ~2 GB.
    knn = KNeighborsClassifier(n_neighbors=KNN_K, metric="cosine", algorithm="brute")
    with config_context(working_memory=KNN_WORKING_MEMORY_MB):
        pred = knn.fit(X_train[:, cols], y_train).predict(X_test[:, cols])
    return pred, True


def split_order(name):
    # "random_07" -> ("random", 7); "prefix" -> ("prefix", -1)
    return ("random", int(name.split("_")[1])) if name.startswith("random_") else (name, -1)


def probe_one(model, dataset, level, readout):
    train = np.load(feature_path(model, dataset, "train"))[readout]
    test = np.load(feature_path(model, dataset, "test"))[readout]
    y_train = DATASETS[dataset]["train"][level].to_numpy()
    y_test = DATASETS[dataset]["test"][level].to_numpy()

    # 1. C at full width
    C, cv_f1 = select_C(train, y_train)

    # 2. standardized coordinates
    scaler = StandardScaler().fit(train)
    z_train, z_test = scaler.transform(train), scaler.transform(test)

    # 3. PCA order (top max(WIDTHS) components), standardized scores
    pca = PCA(n_components=max(WIDTHS), random_state=0).fit(train)
    pca_scaler = StandardScaler().fit(pca.transform(train))
    p_train = pca_scaler.transform(pca.transform(train))
    p_test = pca_scaler.transform(pca.transform(test))

    # 4. logistic-regression probes: (probe, order name, d, X_train, X_test, columns)
    lr_jobs = [("logreg", "full", D_FULL, z_train, z_test, ORDERS["prefix"])]
    for d in WIDTHS:
        for name, cols in ORDERS.items():
            lr_jobs.append(("logreg", name, d, z_train, z_test, cols[:d]))
        lr_jobs.append(("logreg", "pca", d, p_train, p_test, np.arange(d)))

    # 5. kNN check on raw z (prefix and random orders only)
    knn_jobs = [("knn", name, d, train, test, cols[:d])
                for d in KNN_WIDTHS for name, cols in ORDERS.items() if name != "reverse"]

    outputs = Parallel(n_jobs=N_JOBS)(
        delayed(fit_logreg)(Xtr, Xte, cols, y_train, C) for _, _, _, Xtr, Xte, cols in lr_jobs)
    outputs += Parallel(n_jobs=KNN_N_JOBS)(
        delayed(fit_knn)(Xtr, Xte, cols, y_train) for _, _, _, Xtr, Xte, cols in knn_jobs)
    jobs = lr_jobs + knn_jobs

    test_classes = np.unique(y_test)
    rows, preds = [], {}
    for (probe, name, d, *_), (pred, converged) in zip(jobs, outputs):
        order, rep = split_order(name)
        rows.append({
            "model": model, "dataset": dataset, "level": level, "readout": readout,
            "probe": probe, "order": order, "rep": rep, "d": d, "C": C,
            "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0),
            "f1_macro_test_classes": f1_score(y_test, pred, average="macro",
                                              labels=test_classes, zero_division=0),
            "converged": converged,
        })
        preds[f"{probe}__{name}__{d}"] = pred.astype(np.int16)
    np.savez_compressed(PRED_DIR / f"{model}__{dataset}__{level}__{readout}.npz", y_test=y_test, **preds)
    return rows, cv_f1


def run_all(models):
    done = set()
    if RESULT_CSV.exists():
        old = pd.read_csv(RESULT_CSV)
        done = set(zip(old["model"], old["dataset"], old["level"], old["readout"]))
    for model in models:
        for dataset, levels in LEVELS.items():
            for level in levels:
                for readout in READOUTS:
                    if (model, dataset, level, readout) in done:
                        continue
                    start = time.time()
                    rows, cv_f1 = probe_one(model, dataset, level, readout)
                    pd.DataFrame(rows).to_csv(RESULT_CSV, mode="a", header=not RESULT_CSV.exists(), index=False)
                    best_C = rows[0]["C"]
                    print(f"{model:12s} {dataset:5s} {level:9s} {readout:6s} C={best_C:<5g} "
                          f"(CV F1 {cv_f1[best_C]:.3f})  {len(rows)} probes  {time.time() - start:6.1f} s")
    return pd.read_csv(RESULT_CSV)

Run the probes. Each model × dataset × level fits 86 logistic regressions (20 for choosing C, 1 at full width, 13 orders × 5 widths) and 22 kNN probes. GoEmotions (36k train rows) is much slower than TREC.

In [56]:
results = run_all(MODELS_TO_RUN)
print(f"\n{len(results)} rows in {RESULT_CSV.name};",
      f"not converged: {int((~results['converged']).sum())}")


7400 rows in probe_f1.csv; not converged: 0


## 6. F1 tables

Test macro-F1 for each width d. Rows are label level × arm × order. Each value is the **mean over the 3 seeds** (base has one model only). For `random`, each seed is first averaged over its `N_RANDOM` sets.

The first combinations were run with 30 random orders. Order k always uses seed 1000 + k, so their sets 0–9 are the same as in a 10-order run. The table keeps only sets 0 to `N_RANDOM` − 1, so all models use the same random sets.

The 768 column is the `full` probe, repeated in every row as the reference.

The per-seed and per-set values are in `results`. Set `column="f1_macro_test_classes"` for the other F1 value.

In [57]:
ARM_ORDER = ["base", "mrl", "mrl_sia", "mrl_pic", "mipic"]
ORDER_ORDER = ["prefix", "random", "reverse", "pca"]


def f1_table(results, dataset, probe="logreg", readout="pooler", column="f1_macro"):
    r = results[(results["dataset"] == dataset) & (results["probe"] == probe)
                & (results["readout"] == readout)].copy()
    r = r[r["rep"] < N_RANDOM]           # same random sets for all models (rep = -1 for non-random orders)
    r["arm"] = r["model"].str.replace(r"_s\d+$", "", regex=True)

    # copy the full-width score into every order as its d = 768 column
    full = r[r["order"] == "full"]
    present = [o for o in ORDER_ORDER if o in set(r["order"])]
    r = pd.concat([r[r["order"] != "full"]] + [full.assign(order=o) for o in present])

    per_model = r.groupby(["level", "arm", "model", "order", "d"])[column].mean()   # random: mean of N_RANDOM sets
    table = per_model.groupby(["level", "arm", "order", "d"]).mean().unstack("d")    # mean over seeds

    levels = [lv for lv in LEVELS[dataset] if lv in table.index.get_level_values("level")]
    arms = [a for a in ARM_ORDER if a in table.index.get_level_values("arm")]
    return table.reindex(pd.MultiIndex.from_product([levels, arms, present])).dropna(how="all").round(3)


for dataset in LEVELS:
    print(f"\n=== {dataset}: logistic regression, test macro-F1 ===")
    display(f1_table(results, dataset))


=== trec: logistic regression, test macro-F1 ===


d                         16     32     64     128    256    768
coarse base    prefix   0.512  0.638  0.804  0.880  0.918  0.922
               random   0.504  0.690  0.819  0.889  0.914  0.922
               reverse  0.498  0.627  0.815  0.900  0.915  0.922
               pca      0.643  0.782  0.895  0.916  0.921  0.922
       mrl     prefix   0.559  0.706  0.763  0.818  0.883  0.895
               random   0.516  0.669  0.767  0.837  0.871  0.895
               reverse  0.463  0.681  0.782  0.844  0.867  0.895
               pca      0.715  0.766  0.814  0.870  0.893  0.895
       mrl_sia prefix   0.600  0.704  0.776  0.801  0.871  0.897
               random   0.520  0.669  0.767  0.836  0.871  0.897
               reverse  0.451  0.711  0.781  0.855  0.869  0.897
               pca      0.704  0.760  0.829  0.865  0.882  0.897
       mrl_pic prefix   0.556  0.632  0.712  0.743  0.836  0.871
               random   0.450  0.593  0.703  0.771  0.822  0.871
               reverse  0.448  0.585  0.693  0.768  0.808  0.871
               pca      0.608  0.705  0.746  0.803  0.832  0.871
       mipic   prefix   0.547  0.630  0.724  0.781  0.839  0.863
               random   0.466  0.609  0.698  0.787  0.833  0.863
               reverse  0.463  0.613  0.713  0.785  0.826  0.863
               pca      0.636  0.690  0.755  0.799  0.840  0.863
fine   base    prefix   0.171  0.374  0.522  0.630  0.650  0.701
               random   0.210  0.370  0.518  0.592  0.651  0.701
               reverse  0.280  0.391  0.521  0.632  0.636  0.701
               pca      0.271  0.428  0.560  0.598  0.687  0.701
       mrl     prefix   0.244  0.415  0.532  0.607  0.629  0.666
               random   0.223  0.381  0.513  0.587  0.629  0.666
               reverse  0.251  0.381  0.497  0.597  0.643  0.666
               pca      0.301  0.480  0.573  0.597  0.626  0.666
       mrl_sia prefix   0.212  0.383  0.490  0.532  0.635  0.655
               random   0.184  0.332  0.478  0.569  0.630  0.655
               reverse  0.200  0.332  0.504  0.586  0.611  0.655
               pca      0.272  0.395  0.534  0.626  0.632  0.655
       mrl_pic prefix   0.194  0.337  0.398  0.479  0.525  0.563
               random   0.166  0.291  0.404  0.482  0.526  0.563
               reverse  0.167  0.261  0.367  0.469  0.525  0.563
               pca      0.225  0.386  0.474  0.509  0.534  0.563
       mipic   prefix   0.181  0.322  0.415  0.496  0.545  0.570
               random   0.152  0.273  0.401  0.489  0.538  0.570
               reverse  0.162  0.276  0.393  0.485  0.533  0.570
               pca      0.226  0.369  0.474  0.523  0.548  0.570


=== goemo: logistic regression, test macro-F1 ===


d                            16     32     64     128    256    768
sentiment base    prefix   0.519  0.552  0.577  0.601  0.618  0.628
                  random   0.493  0.541  0.579  0.599  0.617  0.628
                  reverse  0.481  0.534  0.573  0.600  0.620  0.628
                  pca      0.573  0.587  0.598  0.617  0.617  0.628
          mrl     prefix   0.491  0.532  0.546  0.575  0.597  0.618
                  random   0.418  0.487  0.543  0.576  0.600  0.618
                  reverse  0.416  0.494  0.535  0.579  0.603  0.618
                  pca      0.516  0.551  0.576  0.598  0.605  0.618
          mrl_sia prefix   0.496  0.527  0.548  0.578  0.598  0.613
                  random   0.425  0.492  0.545  0.574  0.596  0.613
                  reverse  0.401  0.495  0.540  0.572  0.594  0.613
                  pca      0.514  0.552  0.575  0.592  0.602  0.613
          mrl_pic prefix   0.491  0.516  0.531  0.560  0.580  0.599
                  random   0.403  0.470  0.525  0.554  0.577  0.599
                  reverse  0.382  0.456  0.521  0.555  0.575  0.599
                  pca      0.499  0.536  0.557  0.566  0.584  0.599
          mipic   prefix   0.465  0.496  0.519  0.544  0.567  0.592
                  random   0.394  0.455  0.507  0.543  0.568  0.592
                  reverse  0.369  0.463  0.508  0.548  0.567  0.592
                  pca      0.477  0.513  0.543  0.563  0.575  0.592
ekman     base    prefix   0.295  0.377  0.424  0.471  0.510  0.532
                  random   0.289  0.362  0.431  0.472  0.511  0.532
                  reverse  0.282  0.368  0.417  0.479  0.512  0.532
                  pca      0.347  0.412  0.464  0.511  0.521  0.532
          mrl     prefix   0.290  0.363  0.415  0.453  0.495  0.514
                  random   0.237  0.324  0.402  0.452  0.491  0.514
                  reverse  0.231  0.328  0.401  0.460  0.485  0.514
                  pca      0.342  0.410  0.446  0.476  0.495  0.514
          mrl_sia prefix   0.298  0.351  0.403  0.449  0.493  0.510
                  random   0.237  0.324  0.394  0.447  0.482  0.510
                  reverse  0.223  0.330  0.398  0.445  0.474  0.510
                  pca      0.338  0.396  0.430  0.465  0.492  0.510
          mrl_pic prefix   0.276  0.339  0.388  0.440  0.466  0.495
                  random   0.220  0.306  0.384  0.435  0.467  0.495
                  reverse  0.207  0.301  0.387  0.433  0.460  0.495
                  pca      0.307  0.385  0.420  0.450  0.481  0.495
          mipic   prefix   0.268  0.324  0.377  0.428  0.457  0.479
                  random   0.212  0.287  0.361  0.416  0.451  0.479
                  reverse  0.201  0.293  0.368  0.417  0.448  0.479
                  pca      0.298  0.353  0.409  0.437  0.467  0.479
fine      base    prefix   0.112  0.174  0.234  0.305  0.354  0.374
                  random   0.117  0.184  0.256  0.314  0.353  0.374
                  reverse  0.112  0.182  0.256  0.303  0.352  0.374
                  pca      0.168  0.239  0.296  0.336  0.368  0.374
          mrl     prefix   0.121  0.179  0.240  0.275  0.316  0.365
                  random   0.095  0.164  0.230  0.279  0.321  0.365
                  reverse  0.098  0.175  0.239  0.293  0.330  0.365
                  pca      0.168  0.223  0.268  0.298  0.326  0.365
          mrl_sia prefix   0.128  0.177  0.230  0.271  0.318  0.349
                  random   0.096  0.163  0.228  0.278  0.316  0.349
                  reverse  0.097  0.163  0.238  0.292  0.316  0.349
                  pca      0.167  0.221  0.267  0.290  0.321  0.349
          mrl_pic prefix   0.115  0.164  0.205  0.259  0.299  0.331
                  random   0.085  0.145  0.208  0.259  0.296  0.331
                  reverse  0.085  0.141  0.200  0.258  0.294  0.331
                  pca      0.142  0.197  0.241  0.272  0.308  0.331
          mipic   prefix   0.106  0.154  0.194  0.241  0.275  0.318
                  random   0.077  0.132  0.195  0.244  0

In [58]:
for dataset in LEVELS:
    print(f"\n=== {dataset}: kNN check (k = {KNN_K}, cosine, raw z) ===")
    display(f1_table(results, dataset, probe="knn"))


=== trec: kNN check (k = 10, cosine, raw z) ===


d                         16     32
coarse base    prefix  0.659  0.767
               random  0.672  0.770
       mrl     prefix  0.713  0.771
               random  0.659  0.759
       mrl_sia prefix  0.726  0.784
               random  0.671  0.764
       mrl_pic prefix  0.672  0.713
               random  0.598  0.695
       mipic   prefix  0.646  0.715
               random  0.608  0.699
fine   base    prefix  0.211  0.298
               random  0.226  0.303
       mrl     prefix  0.305  0.379
               random  0.280  0.365
       mrl_sia prefix  0.306  0.364
               random  0.269  0.373
       mrl_pic prefix  0.269  0.293
               random  0.228  0.296
       mipic   prefix  0.263  0.329
               random  0.233  0.311


=== goemo: kNN check (k = 10, cosine, raw z) ===


d                            16     32
sentiment base    prefix  0.504  0.531
                  random  0.465  0.504
          mrl     prefix  0.456  0.482
                  random  0.414  0.464
          mrl_sia prefix  0.470  0.497
                  random  0.426  0.470
          mrl_pic prefix  0.473  0.483
                  random  0.420  0.461
          mipic   prefix  0.442  0.454
                  random  0.402  0.441
ekman     base    prefix  0.310  0.363
                  random  0.299  0.341
          mrl     prefix  0.314  0.343
                  random  0.280  0.328
          mrl_sia prefix  0.325  0.354
                  random  0.284  0.331
          mrl_pic prefix  0.316  0.342
                  random  0.283  0.330
          mipic   prefix  0.308  0.315
                  random  0.267  0.310
fine      base    prefix  0.133  0.179
                  random  0.135  0.170
          mrl     prefix  0.148  0.179
                  random  0.130  0.171
          mrl_sia prefix  0.157  0.181
                  random  0.135  0.172
          mrl_pic prefix  0.147  0.169
                  random  0.130  0.167
          mipic   prefix  0.146  0.157
                  random  0.121  0.152

## 7. Metrics (plan §7)

All metrics use the logistic-regression probes on `pooler_output`, at d = 16 and 32 (the widths of the decision rules), and the first `N_RANDOM` random orders.

- **S_chance**: the macro-F1 of a classifier that guesses each label at its train frequency. For each class, precision = test share p and recall = train share q, so F1 = 2pq / (p + q).
- **R(d)** = (S(d) − S_chance) / (S(768) − S_chance). 0 = chance, 1 = as good as all 768 coordinates.
- **A(d)** = S_prefix(d) − mean S_random(d). We also show:
  - `z` = A / SD of the random scores. **The H1-a rule uses z > 2** (plan §8);
  - `beats_all` = the prefix is above all `N_RANDOM` random sets;
  - `percentile` = the % of random sets below the prefix.
- **ΔCFI(d)** = CFI_prefix(d) − mean CFI_random(d), with CFI = R_coarse − R_fine. Random order k is paired over the two levels (the same coordinates).
- **ΔCFI(MIPIC) − ΔCFI(MRL)**: calculated for each seed, MIPIC seed s against MRL seed s.

**Label pairs.** TREC has one pair (coarse–fine). GoEmotions has three (sentiment–fine, ekman–fine, sentiment–ekman). The plan does not say which GoEmotions pair the decision rules use, so all are shown.

In [59]:
METRIC_WIDTHS = [16, 32]                # plan §8: the decision rules use d = 16 and 32
CFI_PAIRS = [("trec", "coarse", "fine"),
             ("goemo", "sentiment", "fine"),
             ("goemo", "ekman", "fine"),
             ("goemo", "sentiment", "ekman")]


def chance_f1(dataset, level):
    # Expected macro-F1 of guessing each label at its train frequency: per class F1 = 2pq / (p + q).
    q = DATASETS[dataset]["train"][level].value_counts(normalize=True)
    p = DATASETS[dataset]["test"][level].value_counts(normalize=True)
    labels = q.index.union(p.index)
    q, p = q.reindex(labels, fill_value=0.0), p.reindex(labels, fill_value=0.0)
    return float((2 * p * q / (p + q)).mean())


S_CHANCE = {(ds, lv): chance_f1(ds, lv) for ds, levels in LEVELS.items() for lv in levels}


def arm_of(model):
    return model.rsplit("_s", 1)[0] if model != "base" else "base"


def seed_of(model):
    return model.rsplit("_s", 1)[1] if model != "base" else "-"


pd.Series(S_CHANCE, name="S_chance").round(3).to_frame()

S_chance
trec  coarse        0.163
      fine          0.016
goemo sentiment     0.250
      ekman         0.143
      fine          0.036

### 7a. Prefix against random (H1-a)

One row for each model, label level and width. The summary table shows the mean A and the mean z for each arm, and also the number of seeds with z > 2 (the H1-a rule).

In [60]:
def prefix_vs_random(probe="logreg"):
    r = results[(results["readout"] == "pooler") & (results["probe"] == probe)
                & (results["rep"] < N_RANDOM) & results["d"].isin(METRIC_WIDTHS)]
    keys = ["model", "dataset", "level", "d"]
    pre = r[r["order"] == "prefix"].set_index(keys)["f1_macro"]
    rnd = r[r["order"] == "random"]
    grp = rnd.groupby(keys)["f1_macro"]
    out = pd.DataFrame({"S_prefix": pre, "random_mean": grp.mean(), "random_sd": grp.std(),
                        "random_max": grp.max()})
    out["A"] = out["S_prefix"] - out["random_mean"]
    out["z"] = out["A"] / out["random_sd"]
    out["beats_all"] = out["S_prefix"] > out["random_max"]
    below = rnd.join(pre.rename("p"), on=keys).assign(b=lambda x: x["f1_macro"] < x["p"])
    out["percentile"] = below.groupby(keys)["b"].mean() * 100
    out = out.reset_index()
    out["arm"], out["seed"] = out["model"].map(arm_of), out["model"].map(seed_of)
    return out


def h1a_summary(table):
    g = table.groupby(["dataset", "level", "d", "arm"])
    z_pass = table.assign(p=table["z"] > 2).groupby(["dataset", "level", "d", "arm"])["p"]
    s = pd.DataFrame({"A": g["A"].mean().round(3), "z": g["z"].mean().round(2),
                      "seeds z > 2": z_pass.sum().astype(int).astype(str) + "/" + z_pass.size().astype(str)})
    return s.unstack("arm").reindex(columns=ARM_ORDER, level=1)


H1A = prefix_vs_random("logreg")
H1A_KNN = prefix_vs_random("knn")
H1A.to_csv(RESULT_DIR / "metrics_h1a.csv", index=False)

print("Logistic regression")
display(h1a_summary(H1A))
print("kNN check")
display(h1a_summary(H1A_KNN))

Logistic regression


A                                   z                \
arm                    base    mrl mrl_sia mrl_pic  mipic  base   mrl mrl_sia   
dataset level     d                                                             
goemo   ekman     16  0.006  0.053   0.061   0.056  0.056  0.42  2.87    3.38   
                  32  0.014  0.039   0.027   0.033  0.038  0.85  2.06    1.53   
        fine      16 -0.005  0.026   0.031   0.030  0.029 -0.54  3.03    3.64   
                  32 -0.011  0.015   0.014   0.019  0.022 -1.46  1.85    1.82   
        sentiment 16  0.026  0.073   0.071   0.087  0.072  2.07  3.69    3.39   
                  32  0.012  0.044   0.036   0.045  0.041  1.17  3.12    2.15   
trec    coarse    16  0.008  0.043   0.080   0.106  0.080  0.27  0.83    1.79   
                  32 -0.053  0.037   0.034   0.039  0.021 -1.99  1.17    1.26   
        fine      16 -0.038  0.022   0.028   0.028  0.030 -1.67  0.71    1.17   
                  32  0.004  0.034   0.051   0.045  0.049  0.09  1.55    1.49   

                                   seeds z > 2                             
arm                  mrl_pic mipic        base  mrl mrl_sia mrl_pic mipic  
dataset level     d                                                        
goemo   ekman     16    3.11  3.18         0/1  3/3     3/3     3/3   3/3  
                  32    1.79  2.13         0/1  1/3     0/3     1/3   2/3  
        fine      16    3.04  3.48         0/1  3/3     3/3     2/3   3/3  
                  32    2.20  2.74         0/1  2/3     1/3     2/3   2/3  
        sentiment 16    3.87  4.01         1/1  3/3     3/3     3/3   3/3  
                  32    2.65  2.58         0/1  3/3     2/3     3/3   3/3  
trec    coarse    16    3.28  2.30         0/1  0/3     0/3     2/3   2/3  
                  32    1.53  0.50         0/1  0/3     0/3     2/3   0/3  
        fine      16    1.08  1.25         0/1  1/3     0/3     0/3   0/3  
                  32    1.64  2.44         0/1  0/3     0/3     2/3   2/3

kNN check


A                                   z                \
arm                    base    mrl mrl_sia mrl_pic  mipic  base   mrl mrl_sia   
dataset level     d                                                             
goemo   ekman     16  0.011  0.034   0.041   0.032  0.042  0.94  2.46    3.36   
                  32  0.022  0.015   0.023   0.011  0.006  2.00  1.63    2.19   
        fine      16 -0.003  0.018   0.022   0.017  0.025 -0.35  2.61    2.77   
                  32  0.009  0.008   0.009   0.002  0.005  0.93  1.62    1.49   
        sentiment 16  0.038  0.042   0.044   0.053  0.040  4.61  3.12    3.67   
                  32  0.027  0.019   0.027   0.021  0.013  2.45  1.83    2.87   
trec    coarse    16 -0.013  0.053   0.055   0.073  0.038 -1.13  1.83    2.93   
                  32 -0.003  0.012   0.020   0.018  0.016 -0.16  0.99    0.95   
        fine      16 -0.016  0.025   0.037   0.041  0.029 -0.67  0.79    1.38   
                  32 -0.005  0.014  -0.009  -0.003  0.018 -0.27  0.47   -0.15   

                                   seeds z > 2                             
arm                  mrl_pic mipic        base  mrl mrl_sia mrl_pic mipic  
dataset level     d                                                        
goemo   ekman     16    2.94  4.93         0/1  2/3     3/3     3/3   3/3  
                  32    1.22  0.56         1/1  1/3     2/3     0/3   0/3  
        fine      16    2.47  4.06         0/1  2/3     2/3     2/3   3/3  
                  32    0.43  0.88         0/1  1/3     1/3     0/3   0/3  
        sentiment 16    4.60  4.57         1/1  3/3     3/3     3/3   3/3  
                  32    2.01  1.60         1/1  1/3     2/3     1/3   0/3  
trec    coarse    16    3.19  1.74         0/1  2/3     3/3     3/3   2/3  
                  32    0.59  0.74         0/1  1/3     0/3     0/3   0/3  
        fine      16    1.54  1.27         0/1  0/3     1/3     1/3   0/3  
                  32    0.07  0.79         0/1  0/3     0/3     0/3   0/3

### 7b. Paired bootstrap and ΔCFI (H1-b)

- **Resampling.** Each resample draws the test rows at random, with replacement: 1,000 resamples, seed 0. The same resamples serve all models, orders and label levels of a dataset. The levels share the same test sentences, so CFI uses the same rows for coarse and fine.
- **Fast macro-F1.** We store each resample as a count of how often it draws each test row. The confusion counts of all 1,000 resamples are then three matrix products, so we do not need 1,000 calls to `f1_score`. Index 0 is the full test set, so its value is the normal point estimate.
- **CI.** The 95% CI is the 2.5th to 97.5th percentile of the 1,000 values. S_chance is kept fixed.

In [61]:
N_BOOT = 1000
BOOT_SEED = 0


def resample_weights(n_rows):
    # Row 0 = full test set (all weights 1); rows 1..N_BOOT = bootstrap draw counts.
    rng = np.random.default_rng(BOOT_SEED)
    w = np.ones((N_BOOT + 1, n_rows), dtype=np.float32)
    for b in range(1, N_BOOT + 1):
        w[b] = np.bincount(rng.integers(0, n_rows, n_rows), minlength=n_rows)
    return w


def macro_f1_weighted(y, pred, w, n_classes):
    # Macro-F1 for every row of w at once. F1_k = 2 TP_k / (n_true_k + n_pred_k); like sklearn's
    # default, the mean is over the classes that occur in the true labels or the predictions.
    Y = np.eye(n_classes, dtype=np.float32)[y]
    P = np.eye(n_classes, dtype=np.float32)[pred]
    tp, n_true, n_pred = w @ (Y * P), w @ Y, w @ P
    denom = n_true + n_pred
    f1 = np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)
    return f1.sum(axis=1) / (denom > 0).sum(axis=1)


def bootstrap_scores(models):
    # {(model, dataset, level, order, d): array [N_BOOT + 1]}, order in prefix / random_kk / full
    keys = ["logreg__full__768"] + [f"logreg__{o}__{d}" for d in METRIC_WIDTHS
                                    for o in ["prefix"] + [f"random_{k:02d}" for k in range(N_RANDOM)]]
    scores = {}
    for ds, levels in LEVELS.items():
        w = resample_weights(len(DATASETS[ds]["test"]))
        for model in models:
            for lv in levels:
                saved = np.load(PRED_DIR / f"{model}__{ds}__{lv}__pooler.npz")
                y = saved["y_test"].astype(int)
                n_classes = int(DATASETS[ds]["train"][lv].max()) + 1
                for key in keys:
                    _, order, d = key.split("__")
                    scores[(model, ds, lv, order, int(d))] = macro_f1_weighted(
                        y, saved[key].astype(int), w, n_classes)
    return scores


start = time.time()
BOOT = bootstrap_scores(MODELS_TO_RUN)
print(f"{len(BOOT)} score arrays in {time.time() - start:.0f} s")

# check: index 0 (full test set) must equal the saved sklearn macro-F1
chk = results[(results["probe"] == "logreg") & (results["readout"] == "pooler")
              & (results["order"] == "prefix") & (results["d"] == 16)]
gap = max(abs(BOOT[(m, ds, lv, "prefix", 16)][0] - f) for m, ds, lv, f in
          chk[["model", "dataset", "level", "f1_macro"]].itertuples(index=False))
print(f"largest difference from sklearn f1_macro: {gap:.2e}")

1495 score arrays in 17 s
largest difference from sklearn f1_macro: 4.46e-08


In [62]:
def retention(model, ds, level, order, d):
    ch = S_CHANCE[(ds, level)]
    return (BOOT[(model, ds, level, order, d)] - ch) / (BOOT[(model, ds, level, "full", 768)] - ch)


def cfi(model, ds, coarse, fine, order, d):
    return retention(model, ds, coarse, order, d) - retention(model, ds, fine, order, d)


def delta_cfi(model, ds, coarse, fine, d):
    rand = np.mean([cfi(model, ds, coarse, fine, f"random_{k:02d}", d) for k in range(N_RANDOM)], axis=0)
    return cfi(model, ds, coarse, fine, "prefix", d) - rand


def ci_row(values):
    lo, hi = np.percentile(values[1:], [2.5, 97.5])
    return {"value": values[0], "ci_low": lo, "ci_high": hi, "ci_above_0": lo > 0, "ci_below_0": hi < 0}


rows = []
for ds, coarse, fine in CFI_PAIRS:
    for d in METRIC_WIDTHS:
        for model in MODELS_TO_RUN:
            rows.append({"pair": f"{ds}: {coarse}-{fine}", "d": d, "model": model, "arm": arm_of(model),
                         "seed": seed_of(model), **ci_row(delta_cfi(model, ds, coarse, fine, d))})
DCFI = pd.DataFrame(rows)

rows = []
for ds, coarse, fine in CFI_PAIRS:
    for d in METRIC_WIDTHS:
        for seed in SEEDS:
            diff = delta_cfi(f"mipic_s{seed}", ds, coarse, fine, d) - delta_cfi(f"mrl_s{seed}", ds, coarse, fine, d)
            rows.append({"pair": f"{ds}: {coarse}-{fine}", "d": d, "seed": seed, **ci_row(diff)})
DCFI_MIPIC_VS_MRL = pd.DataFrame(rows)

DCFI.to_csv(RESULT_DIR / "metrics_dcfi.csv", index=False)
DCFI_MIPIC_VS_MRL.to_csv(RESULT_DIR / "metrics_dcfi_mipic_vs_mrl.csv", index=False)


def fmt(r):
    return f"{r.value:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}]"


print("ΔCFI with 95% CI (one column per seed)")
display(DCFI.assign(cell=DCFI.apply(fmt, axis=1))
        .pivot_table(index=["pair", "d", "arm"], columns="seed", values="cell", aggfunc="first")
        .reindex(ARM_ORDER, level="arm").fillna(""))     # base has no seed, so it sits in column "-"
print("ΔCFI(MIPIC) − ΔCFI(MRL) with 95% CI")
display(DCFI_MIPIC_VS_MRL.assign(cell=DCFI_MIPIC_VS_MRL.apply(fmt, axis=1))
        .pivot_table(index=["pair", "d"], columns="seed", values="cell", aggfunc="first"))

ΔCFI with 95% CI (one column per seed)


seed                                                     -  \
pair                   d  arm                                
goemo: ekman-fine      16 base     +0.030 [-0.006, +0.066]   
                          mrl                                
                          mrl_sia                            
                          mrl_pic                            
                          mipic                              
                       32 base     +0.068 [+0.019, +0.112]   
                          mrl                                
                          mrl_sia                            
                          mrl_pic                            
                          mipic                              
goemo: sentiment-ekman 16 base     +0.054 [+0.018, +0.088]   
                          mrl                                
                          mrl_sia                            
                          mrl_pic                            
                          mipic                              
                       32 base     -0.007 [-0.050, +0.038]   
                          mrl                                
                          mrl_sia                            
                          mrl_pic                            
                          mipic                              
goemo: sentiment-fine  16 base     +0.084 [+0.043, +0.122]   
                          mrl                                
                          mrl_sia                            
                          mrl_pic                            
                          mipic                              
                       32 base     +0.062 [+0.023, +0.096]   
                          mrl                                
                          mrl_sia                            
                          mrl_pic                            
                          mipic                              
trec: coarse-fine      16 base     +0.066 [-0.035, +0.151]   
                          mrl                                
                          mrl_sia                            
                          mrl_pic                            
                          mipic                              
                       32 base     -0.076 [-0.165, -0.004]   
                          mrl                                
                          mrl_sia                            
                          mrl_pic                            
                          mipic                              

seed                                                     1  \
pair                   d  arm                                
goemo: ekman-fine      16 base                               
                          mrl      +0.100 [+0.048, +0.150]   
                          mrl_sia  +0.049 [-0.006, +0.100]   
                          mrl_pic  +0.046 [-0.003, +0.094]   
                          mipic    +0.052 [+0.012, +0.095]   
                       32 base                               
                          mrl      +0.049 [-0.004, +0.104]   
                          mrl_sia  +0.041 [-0.008, +0.089]   
                          mrl_pic  +0.008 [-0.047, +0.063]   
                          mipic    +0.031 [-0.019, +0.078]   
goemo: sentiment-ekman 16 base                               
                          mrl      +0.033 [-0.017, +0.082]   
                          mrl_sia  +0.011 [-0.036, +0.058]   
                          mrl_pic  +0.073 [+0.033, +0.107]   
                          mipic    +0.091 [+0.051, +0.127]   
                       32 base                               
                          mrl      +0.022 [-0.031, +0.076]   
                          mrl_sia  +0.013 [-0.031, +0.059]   
                          mrl_pic  +0.025 [-0.024, +0.075]   
                          mipic    +0.032 [-0.016, +0.072]   
goemo: sentiment-fine  16 base 

ΔCFI(MIPIC) − ΔCFI(MRL) with 95% CI


seed                                            1                        2   \
pair                   d                                                      
goemo: ekman-fine      16  -0.048 [-0.102, +0.008]  +0.017 [-0.048, +0.078]   
                       32  -0.018 [-0.083, +0.047]  -0.016 [-0.093, +0.060]   
goemo: sentiment-ekman 16  +0.058 [+0.006, +0.115]  -0.010 [-0.060, +0.046]   
                       32  +0.010 [-0.052, +0.070]  -0.010 [-0.077, +0.061]   
goemo: sentiment-fine  16  +0.010 [-0.047, +0.066]  +0.008 [-0.053, +0.072]   
                       32  -0.007 [-0.062, +0.051]  -0.026 [-0.089, +0.038]   
trec: coarse-fine      16  -0.030 [-0.182, +0.073]  +0.068 [-0.033, +0.187]   
                       32  -0.119 [-0.239, +0.036]  -0.013 [-0.123, +0.116]   

seed                                            42  
pair                   d                            
goemo: ekman-fine      16  +0.025 [-0.041, +0.087]  
                       32  -0.045 [-0.102, +0.014]  
goemo: sentiment-ekman 16  -0.081 [-0.140, -0.025]  
                       32  -0.024 [-0.073, +0.028]  
goemo: sentiment-fine  16  -0.056 [-0.118, +0.004]  
                       32  -0.068 [-0.124, -0.008]  
trec: coarse-fine      16  +0.074 [-0.054, +0.229]  
                       32  -0.037 [-0.175, +0.080]

## 8. Decision rules (plan §8)

The rules use d = 16 and 32. A part is supported only if it passes on both datasets and in all 3 seeds. Each cell shows how many MIPIC seeds pass.

| Part | Supported if |
|---|---|
| H1-a | z > 2 for MIPIC in every seed (more than 2 SD above the mean of the 10 random sets), and z ≤ 2 for base |
| H1-b | the 95% CI of ΔCFI for MIPIC is above 0 in every seed |
| H1-b, caused by MIPIC | the 95% CI of ΔCFI(MIPIC) − ΔCFI(MRL) is above 0 in every seed |

**Why z > 2 for H1-a.** The first rule was "above the 95th percentile of the random sets". With 10 random sets, the highest possible result is "above all 10", which is about the 91st percentile, so that rule could not pass. z > 2 is the nearest equivalent of the 95% level. We made this change because of the number of random sets, not because of the results.

The base columns are the null check. Base has no order, so it must not pass.

In [63]:
def count(mask):
    return f"{int(mask.sum())}/{int(mask.size)}"


# H1-a: per dataset x level x d
rows = []
for (ds, lv, d), g in H1A.groupby(["dataset", "level", "d"], sort=False):
    mip, base = g[g["arm"] == "mipic"], g[g["arm"] == "base"]
    rows.append({"dataset": ds, "level": lv, "d": d,
                 "MIPIC z (per seed)": ", ".join(f"{z:.1f}" for z in mip["z"]),
                 "MIPIC z > 2": count(mip["z"] > 2), "base z": round(float(base["z"].iloc[0]), 1),
                 "pass H1-a": bool((mip["z"] > 2).all() and not (base["z"] > 2).any())})
RULE_A = pd.DataFrame(rows)

# H1-b: per pair x d
rows = []
for (pair, d), g in DCFI.groupby(["pair", "d"], sort=False):
    mip, base = g[g["arm"] == "mipic"], g[g["arm"] == "base"]
    diff = DCFI_MIPIC_VS_MRL[(DCFI_MIPIC_VS_MRL["pair"] == pair) & (DCFI_MIPIC_VS_MRL["d"] == d)]
    rows.append({"pair": pair, "d": d,
                 "MIPIC ΔCFI CI > 0": count(mip["ci_above_0"]),
                 "base ΔCFI CI excludes 0": count(base["ci_above_0"] | base["ci_below_0"]),
                 "pass H1-b": bool(mip["ci_above_0"].all()),
                 "MIPIC − MRL CI > 0": count(diff["ci_above_0"]),
                 "pass H1-b (caused by MIPIC)": bool(diff["ci_above_0"].all())})
RULE_B = pd.DataFrame(rows)

print("H1-a")
display(RULE_A)
print("H1-b")
display(RULE_B)

H1-a


,dataset,level,d,MIPIC z (per seed),MIPIC z > 2,base z,pass H1-a
0,goemo,ekman,16,"2.7, 3.2, 3.7",3/3,0.4,True
1,goemo,ekman,32,"1.5, 2.0, 2.9",2/3,0.9,False
2,goemo,fine,16,"2.8, 3.4, 4.2",3/3,-0.5,True
3,goemo,fine,32,"1.6, 3.3, 3.3",2/3,-1.5,False
4,goemo,sentiment,16,"4.1, 3.9, 4.0",3/3,2.1,False
5,goemo,sentiment,32,"2.0, 2.2, 3.5",3/3,1.2,True
6,trec,coarse,16,"0.3, 2.4, 4.2",2/3,0.3,False
7,trec,coarse,32,"-0.3, 0.6, 1.2",0/3,-2.0,False
8,trec,fine,16,"1.4, 0.6, 1.8",0/3,-1.7,False
9,trec,fine,32,"2.9, 0.9, 3.5",2/3,0.1,False


H1-b


,pair,d,MIPIC ΔCFI CI > 0,base ΔCFI CI excludes 0,pass H1-b,MIPIC − MRL CI > 0,pass H1-b (caused by MIPIC)
0,trec: coarse-fine,16,1/3,0/1,False,0/3,False
1,trec: coarse-fine,32,0/3,1/1,False,0/3,False
2,goemo: sentiment-fine,16,2/3,1/1,False,0/3,False
3,goemo: sentiment-fine,32,1/3,1/1,False,0/3,False
4,goemo: ekman-fine,16,3/3,0/1,True,0/3,False
5,goemo: ekman-fine,32,0/3,1/1,False,0/3,False
6,goemo: sentiment-ekman,16,2/3,1/1,False,1/3,False
7,goemo: sentiment-ekman,32,0/3,0/1,False,0/3,False


## 9. Analysis

**Verdict: H1 is not supported.** The MRL prefix is better than random coordinates on GoEmotions. But it does not reliably keep coarse information first, and SIA and PIC do not cause any ordering.

The rules use `pooler_output`, d = 16 and 32, all 3 seeds and both datasets. There are 10 random orders and 1,000 paired bootstrap resamples.

| Part | Result | Main evidence |
|---|---|---|
| **H1-a**: prefix better than random (z > 2) | **Not supported** (passes on GoEmotions only, in 3 of 6 cells) | GoEmotions: MIPIC has z > 2 in 3/3 seeds in 4 of 6 cells (z ≈ 2–4). One of these 4 fails because base also has z > 2. TREC: no cell passes (0–2 of 3 seeds). |
| **H1-b**: coarse first (ΔCFI > 0) | **Not supported** | Only 1 of 8 pair × width cells passes (GoEmotions ekman–fine, d = 16). TREC d = 32 is negative for most arms. |
| **H1-b**: caused by MIPIC | **Not supported** | ΔCFI(MIPIC) − ΔCFI(MRL) has a CI above 0 in 1 of 24 seed cells, and it is significantly *negative* in 2 cells. |

**Points to note**

1. **The prefix effect comes from MRL, not from SIA or PIC.** All four trained arms have almost the same prefix advantage A (about +0.03 to +0.09 F1 at d = 16 on GoEmotions). MIPIC is not better than MRL alone.
2. **The base model fails the null check for ΔCFI.** Base has no order, so its ΔCFI must be about 0. But its CI excludes 0 in 5 of 8 cells: it is positive on GoEmotions and negative on TREC d = 32. Thus a positive ΔCFI can occur without training. The plan says to find the cause before we use ΔCFI. Possible causes:
   - base is one model only, with no seeds;
   - 10 random sets give a noisy random mean;
   - how R scales tasks with different chance levels.
3. **Base also passes the H1-a test in one cell** (GoEmotions sentiment, d = 16: z = 2.1). Thus that cell fails the null check. With 10 cells, one false pass at about the 95% level is possible by chance.
4. **TREC is too small for clear results.** It has 500 test rows. The ΔCFI CIs are about ±0.1 wide, so most TREC cells cannot pass or fail with confidence.
5. **kNN agrees at d = 16 and is weaker at d = 32.** So the prefix advantage at d = 16 does not come from the fitted probe.
6. **Side result:** at full width (768), the untrained base model has the highest F1 on all five label levels (for example, TREC fine 0.70 against 0.57 for MIPIC). MIPIC training does not improve these probe scores. This agrees with Finding 1 of `h11_probe`.

**Open items (these do not change the verdict above)**

- **Choose the GoEmotions pair** for H1-b. H1-b fails with every pair.